# Anchor miner testing notebook

In [17]:
import sys, os
import subprocess
sys.path.append(os.path.dirname(os.getcwd()))
from src.standartize_hlas import normalize_allele
import pandas as pd
import numpy as np
from src import AnchorMiner
from src.predict_anchors import predict_anchors
import tqdm
import re
import matplotlib.pyplot as plt
import sklearn

## 1. Testing how much data will be excluded on a random dataset

In [2]:
df = pd.read_csv('../datasets/epitope_HLA_data.tsv', sep = '\t')[['Peptide', 'HLA_Allele']]
df.drop_duplicates(subset=['Peptide'], keep='first', inplace=True)

HLA_Alleles_standartized = [normalize_allele(i) for i in df['HLA_Allele'].tolist()]
df['HLA_Allele'] = HLA_Alleles_standartized

df = df.drop(df[df['Peptide'].isna()].index)
df = df.drop(df[df['HLA_Allele'].isna()].index)

anchors = []
anchors_masked = []
not_anchors_masked = []
excluded_combos = []
included_combos = []

for index,row in tqdm.tqdm(df.iterrows(), total = len(df)):
   try:
      anchors.append(predict_anchors(row['Peptide'], row['HLA_Allele'], 0.5, 'False')['coords'])
   
      anchors_masked.append(''.join('X' if i in anchors[-1] else char for i, char in enumerate(row['Peptide'])))
      not_anchors_masked.append(''.join('X' if i not in anchors[-1] else char for i, char in enumerate(row['Peptide'])))
      included_combos.append(tuple([row['HLA_Allele'], len(row['Peptide'])]))
   except:
      anchors.append(np.nan)
      anchors_masked.append(np.nan)
      not_anchors_masked.append(np.nan)
      excluded_combos.append(tuple([row['HLA_Allele'], len(row['Peptide'])]))


df['anchors'] = anchors
df['anchors_masked'] = anchors_masked
df['not_anchors_masked'] = not_anchors_masked
    
display(df)




100%|██████████| 19367/19367 [00:42<00:00, 458.76it/s]


,Peptide,HLA_Allele,anchors,anchors_masked,not_anchors_masked
1,AAAAAAAAK,HLA-A03:01,"[1, 8]",AXAAAAAAX,XAXXXXXXK
3,AAAAAAAAL,HLA-B07:01,NaN,NaN,NaN
4,AAAAAAAALY,HLA-A29:02,"[1, 9]",AXAAAAAALX,XAXXXXXXXY
5,AAAAAIFVI,HLA-B51:01,"[1, 8]",AXAAAIFVX,XAXXXXXXI
6,AAAAGWQTL,HLA-A02:01,"[1, 8]",AXAAGWQTX,XAXXXXXXL
...,...,...,...,...,...
62155,SSARSQSER,HLA-A11:01,"[1, 8]",SXARSQSEX,XSXXXXXXR
62162,SMSMILVGV,HLA-A02:01,[8],SMSMILVGX,XXXXXXXXV
62171,YLVGNVCIL,HLA-A02:01,"[1, 8]",YXVGNVCIX,XLXXXXXXL
62210,FLCLLIPGL,HLA-A02:01,"[1, 8]",FXCLLIPGX,XLXXXXXXL


In [3]:
print(f'Number of excluded rows: {df['anchors'].isna().sum()} out of {len(df)} ({round(df['anchors'].isna().sum() / len(df) * 100,1)}%)')
excluded_combos = set(excluded_combos)
print(f'Number of unique excluded HLA-len combos: {len(set(excluded_combos))}')
print('Among them:')
print(f'len(Epitope) < 9 (cannot predict binding): {sum(1 for _, num in excluded_combos if num < 9)}')
print(f'len(Epitope) > 9 (likely too low affinity to epitopes for netmhcpan detection): {sum(1 for _, num in excluded_combos if num > 9)}')
print('Other cases (Alleles currently not recognized by anchor miner due to limited PWM data):')
for i in excluded_combos:
    if i[1] == 9:
        print(i)

Number of excluded rows: 635 out of 19367 (3.3%)
Number of unique excluded HLA-len combos: 89
Among them:
len(Epitope) < 9 (cannot predict binding): 63
len(Epitope) > 9 (likely too low affinity to epitopes for netmhcpan detection): 16
Other cases (Alleles currently not recognized by anchor miner due to limited PWM data):
('HLA-B07:01', 9)
('HLA-B62:01', 9)
('HLA-A33:02', 9)
('HLA-A02:01', 9)
('HLA-A28:01', 9)
('HLA-A35:01', 9)
('HLA-B61:01', 9)
('HLA-B44:01', 9)
('HLA-A24:01', 9)
('HLA-A51:01', 9)


## 2. Testing on structural data

In [46]:
import pandas as pd
import numpy as np
import glob
 
STRUCT_DIR = '/home/aalarkin/Work/anchorminer/datasets/structural_data'



def process_structural_data(path):
    """
    Extract structural anchor positions from HLA contact frequency data.
 
    Args:
        path: Path to contacts CSV file.
        threshold: Mean HLA contact frequency threshold for anchor calling.
                   Recommended: 0.06 based on observed distribution gap.
 
    Returns:
        tuple: (peptide string, freqs dict, anchor positions list 0-indexed)
    """
    df = pd.read_csv(path)
    hla = df[df['region'].str.contains('hla_')]
    threshold = np.mean(hla['contact_freq'])
 
    stats = (hla.groupby('pep_res')['contact_freq']
                .mean()
                .reset_index())

    stats['pos'] = stats['pep_res'].str[1:].astype(int) - 1
    stats['aa']  = stats['pep_res'].str[0]
    stats = stats.sort_values('pos').reset_index(drop=True)
   # print(stats)
 
    peptide = ''.join(stats['aa'].tolist())
 
    freqs = {}
    for _, row in stats.iterrows():
        freqs[str(row['pos'])] = round(float(row['contact_freq']), 3)
 
    anchors = [int(row['pos']) 
               for _, row in stats.iterrows()
               if row['contact_freq'] >= (threshold) * 1.2]
 
    return peptide, freqs, anchors
 

def compare_results(seq, anchors_true, anchors_pred):
    v_true = np.zeros(len(seq))
    v_pred = np.zeros(len(seq))

    for i in anchors_true:
        v_true[i] = 1

    for j in anchors_pred:
        v_pred[j] = 1

    return(round(sklearn.metrics.jaccard_score(v_true, v_pred),3))

 
files = sorted(glob.glob(f'{STRUCT_DIR}/contacts_*.csv'))

 #ALLELES LIST FROM PDB

alleles = {
    '1ao7': 'HLA-A02:01',  # https://www.rcsb.org/structure/1AO7
    '1qrn': 'HLA-A02:01',  # https://www.rcsb.org/structure/1QRN (just HLA-A02 → A02:01)
    '2f53': 'HLA-A02:01',  # https://www.rcsb.org/structure/2F53 (just HLA-A → A02:01)
    '2vlj': 'HLA-A02:01',  # https://www.rcsb.org/structure/2VLJ
    '3hg1': 'HLA-A02:01',  # https://www.rcsb.org/structure/3HG1
    '3o4l': 'HLA-A02:01',  # https://www.rcsb.org/structure/3O4L (just HLA-A → A02:01)
    '3uts': 'HLA-A02:01',  # https://www.rcsb.org/structure/3UTS
    '4mji': 'HLA-B51:01', # https://www.rcsb.org/structure/4MJI
    '4qrp': 'HLA-B08:01',  # https://www.rcsb.org/structure/4QRP
    '5brz': 'HLA-A01:01',  # https://www.rcsb.org/structure/5BRZ (just A1 → A01:01)
    '5bs0': 'HLA-A01:01',  # https://www.rcsb.org/structure/5BS0 (just A1 → A01:01)
    '5hho': 'HLA-A02:01',  # https://www.rcsb.org/structure/5HHO
    '5hyj': 'HLA-A02:01',  # https://www.rcsb.org/structure/5HYJ (just A02 → A02:01)
    '6eqb': 'HLA-A02:01',  # https://www.rcsb.org/structure/6EQB (just A2 → A02:01)
    '6rp9': 'HLA-A02:01',  # https://www.rcsb.org/structure/6RP9
    '7dzn': 'HLA-B42:01',  # https://www.rcsb.org/structure/7DZN
}



structural_results = {}
for path in files:
    name = path.split('/')[-1].replace('contacts_', '').replace('.csv', '')
    peptide, freqs, anchors_struct = process_structural_data(path)
    structural_results[name] = {
        'peptide': peptide,
        'freqs': freqs,
        'anchors_struct': anchors_struct
    }
   # print(f'{name}  {peptide}')
   # print(f'  freqs:   {freqs}')
   # print(f'  anchors struct: {anchors_struct}')
    try:
        am = predict_anchors(peptide, alleles[name], 0.5, 'False')['coords']
    except:
        am = [1, len(peptide)-1]

    structural_results[name]['anchors_pred'] = am
    #print(f'  anchord pred: {am}')

val_df = pd.DataFrame(structural_results).transpose()[['peptide','anchors_struct', 'anchors_pred']]


jac = []
for index, row in val_df.iterrows():
    jac.append(compare_results(row['peptide'], row['anchors_struct'], row['anchors_pred']))

val_df['jaccard'] = np.array(jac)

display(val_df)

print(np.mean(val_df['jaccard']))

,peptide,anchors_struct,anchors_pred,jaccard
1ao7,LLFGYPVYV,"[0, 1, 8]","[1, 8]",0.667
1qrn,LLFGYAVYV,"[0, 1, 2, 8]","[1, 8]",0.500
2f53,SLLMWITQC,"[0, 1, 8]",[1],0.333
2vlj,GILGFVFTL,"[1, 8]","[1, 8]",1.000
3hg1,ELAGIGILTV,"[1, 8, 9]","[1, 9]",0.667
3o4l,GLCTLVAML,"[0, 1, 8]","[1, 8]",0.667
3uts,ALWGPDPAAA,"[0, 1, 2]","[1, 9]",0.250
4mji,TAFTIPSI,"[0, 2, 4]","[1, 7]",0.000
4qrp,HSKKKCDEL,"[0, 2, 4, 8]","[4, 8]",0.500
5brz,EVDPIGHLY,"[0, 1, 8]","[1, 2, 8]",0.500


0.4868947368421052
